In [1]:
import os
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import sys

import glob
import h5py
import torch
import numpy as np
import matplotlib.pyplot as plt
import sys

from torch.utils.data import Dataset
from typing import Optional, Callable, List, Dict, Tuple, Any
from tqdm import tqdm
import json

torch.cuda.empty_cache()

In [2]:
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/")
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/transforms")

In [3]:
from fixed_dataset import FixedDataset
from seg_recon_vit3d import SegRecon_ViT_3D
from transforms.factory import transform_factory

/home/ana-caznok/software/src/miniforge3/envs/agrvai/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# --------------------- CONFIG ---------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_PATH = "/media/ana-caznok/SSD-08/icasp_4090/icasp/data/Link_2"  # TODO: change to actual dataset path
PREPROCESSING = "h5py"  # or None or "downsampled"
FOLD = 0
BATCH_SIZE = 2
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
SAVE_PATH = "./seg_rec_model.pth"
TRANSFORM = "downs4_fft_D40_x_gpu"

In [5]:
# ------------------ DATASET + LOADER ------------------
train_dataset = FixedDataset(
    mode="train",
    base_path=BASE_PATH,
    transform=transform_factory(TRANSFORM),  # You can define transforms here
    preprocessing=PREPROCESSING
)

val_dataset = FixedDataset(
    mode="val",
    base_path=BASE_PATH,
    transform=transform_factory(TRANSFORM),
    preprocessing=PREPROCESSING
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

Loading from hdf files
Number of face masks 234. 

Checking face mask integrity...: 234it [00:00, 483005.48it/s]


initialized 234 images train DatasetRaw with preprocessing: h5py in /media/ana-caznok/SSD-08/icasp_4090/icasp/data/Link_2 with transform 
0: RGB2Pseudo_Hyp with: /media/ana-caznok/SSD-08/recon-segment/ and camera D40
1: Downsample by a factor of 4
2: FourierSpectralTransform(norm=minmax, transform cube=False, device=cuda). Fold: None
Loading from hdf files
Number of face masks 30. 

Checking face mask integrity...: 30it [00:00, 767250.73it/s]

initialized 30 images val DatasetRaw with preprocessing: h5py in /media/ana-caznok/SSD-08/icasp_4090/icasp/data/Link_2 with transform 
0: RGB2Pseudo_Hyp with: /media/ana-caznok/SSD-08/recon-segment/ and camera D40
1: Downsample by a factor of 4
2: FourierSpectralTransform(norm=minmax, transform cube=False, device=cuda). Fold: None


In [6]:

# ------------------ MODEL ------------------
model = SegRecon_ViT_3D(C_input=31, total_channels=61).to(DEVICE)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [7]:

# ------------------ TRAIN LOOP ------------------
for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0
    for x, y, meta in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} - Training"):
        
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # ------------------ EVAL LOOP ------------------
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y, meta in tqdm(val_loader, desc="Validation"):
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            output = model(x)
            loss = criterion(output, y)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}")

# ------------------ SAVE MODEL ------------------
torch.save(model.state_dict(), SAVE_PATH)
print(f"Model saved to {SAVE_PATH}")

Validation: 100%|██████████| 15/15 [02:05<00:00,  8.34s/it]


Epoch 1/10: Train Loss = 0.0301, Val Loss = 0.0263


Epoch 2/10 - Training:   2%|▏         | 2/117 [00:23<22:19, 11.65s/it]


KeyboardInterrupt: 

In [ ]:
x.size()

In [ ]:
output.size()